# graphobs — Quickstart

This notebook walks through the core idea in two acts:

| Act | What you need | What you see |
|-----|---------------|--------------|
| **Act 1** | Nothing — zero credentials | Curated vs messy span contrast inside the notebook |
| **Act 2** | A `.env` with your platform credentials | The same spans in your observability platform |

**The central idea:** state, logs, and traces are three different questions.
A `NodeContract` answers all three with one declaration — it defines which
state paths a node may read and write, and that boundary becomes both the
trace payload and the write validator.

```
State  = what the next node needs      → tight, typed contract
Logs   = what on-call needs            → discrete, redacted, emitted once
Traces = what eval needs               → clean span kinds, curated I/O
```

**Prerequisites:** Python 3.11+.

## Install

The `[demo]` extra pulls in the OpenTelemetry SDK, Arize Phoenix, and the
JupyterLab runtime. Run this cell once, then restart the kernel.

In [ ]:
%pip install -q "graphobs[demo]"

## Imports

In [ ]:
import json
from collections.abc import Mapping
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

from graphobs import NodeContract, add_contract_node, contract_node
from graphobs.contracts.models import StateContractError
from graphobs.demo import configure_local_tracing, span_records
from graphobs.tracing import (
    TracePayloadMode,
    set_span_output,
    start_graph_span,
)

## Define the graph state

The contract uses *dotted paths* to name individual nested keys.
`reads=("request.text",)` means the node may only see
`state["request"]["text"]`, not the rest of `state["request"]`.

Everything under `scratch` is private — it never appears in a span.

In [ ]:
# Synthetic documents — swap node bodies with real LLM/retriever calls.
DOCUMENTS = [
    {
        "title": "Observability Notes",
        "summary": "Contracts keep trace payloads focused.",
    },
    {
        "title": "Graph Debugging",
        "summary": "Logs, traces, and state answer different questions.",
    },
]

INITIAL_STATE = {
    "request": {
        "text": "How do contracts improve graph traces?",
        # Outside every contract — must never appear in a span.
        "raw_notes": "internal pre-processing notes",
    },
    "scratch": {"draft_query": "contract graph trace payload boundaries"},
}


class RagState(TypedDict, total=False):
    request: dict
    classification: dict
    documents: list
    answer: dict
    scratch: dict

## Define the node functions

Plain functions — no telemetry code anywhere. Swap the bodies with real
LLM or retriever calls when moving to production.

In [ ]:
def classify_intent(state: Mapping[str, object]) -> dict[str, object]:
    # swap with a real LLM call
    text = str((state.get("request") or {}).get("text", ""))
    intent = "question" if "?" in text else "statement"
    return {"classification": {"intent": intent}}


def retrieve_docs(state: Mapping[str, object]) -> dict[str, object]:
    # swap with a real retriever call
    text = str((state.get("request") or {}).get("text", "")).lower()
    matching = [
        {"title": d["title"], "summary": d["summary"]}
        for d in DOCUMENTS
        if "trace" in text or "contract" in text
    ]
    return {"documents": matching[:2]}


def answer_question(state: Mapping[str, object]) -> dict[str, object]:
    # swap with a real LLM call
    docs = state.get("documents") or []
    summary = docs[0].get("summary", "No summary") if docs else "No matching documents"
    return {"answer": {"text": f"Based on docs: {summary}"}}

---

## Act 1 — zero credentials: see curated vs messy spans inline

We run the same three-node graph twice:

- **Raw** — no contracts; the entire state blob lands in every span.
- **Contract-wrapped** — each node declares `reads`/`writes`;
  only those paths appear in the span.

The in-memory tracer captures everything locally — no collector required.

In [ ]:
# One call — installs the global TracerProvider for this kernel.
exporter = configure_local_tracing()

### Raw graph — the messy baseline

In [ ]:
raw_graph = StateGraph(RagState)
raw_graph.add_node("classify_intent", classify_intent)
raw_graph.add_node("retrieve_docs", retrieve_docs)
raw_graph.add_node("answer_question", answer_question)
raw_graph.add_edge(START, "classify_intent")
raw_graph.add_edge("classify_intent", "retrieve_docs")
raw_graph.add_edge("retrieve_docs", "answer_question")
raw_graph.add_edge("answer_question", END)

with start_graph_span(
    "raw_rag", "CHAIN", input=INITIAL_STATE, mode=TracePayloadMode.FULL
) as span:
    raw_result = raw_graph.compile().invoke(dict(INITIAL_STATE))
    set_span_output(span, raw_result, mode=TracePayloadMode.FULL)

raw_spans = span_records(exporter)
exporter.clear()

print("RAW spans — notice the full state blob in every input/output:")
print(json.dumps(raw_spans, indent=2))

### Contract-wrapped graph — the curated result

In [ ]:
contract_graph = StateGraph(RagState)

add_contract_node(
    contract_graph,
    NodeContract(
        name="classify_intent",
        reads=("request.text",),
        writes=("classification.intent",),
        span_kind="CHAIN",
    ),
    classify_intent,
)
add_contract_node(
    contract_graph,
    NodeContract(
        name="retrieve_docs",
        reads=("request.text", "classification.intent"),
        writes=("documents",),
        span_kind="RETRIEVER",
    ),
    retrieve_docs,
)
add_contract_node(
    contract_graph,
    NodeContract(
        name="answer_question",
        reads=("request.text", "documents"),
        writes=("answer.text",),
        span_kind="CHAIN",
    ),
    answer_question,
)
contract_graph.add_edge(START, "classify_intent")
contract_graph.add_edge("classify_intent", "retrieve_docs")
contract_graph.add_edge("retrieve_docs", "answer_question")
contract_graph.add_edge("answer_question", END)

contract_result = contract_graph.compile().invoke(dict(INITIAL_STATE))
contract_spans = span_records(exporter)
exporter.clear()

print("CONTRACT spans — only declared paths; raw_notes and scratch never appear:")
print(json.dumps(contract_spans, indent=2))

### Undeclared write — the contract catches it

If a node writes a key it never declared, `StateContractError` fires.
Crucially, the error stores *only path names* — never state values — so
no PII leaks through the exception.

In [ ]:
def sneaky_retriever(state: Mapping[str, object]) -> dict[str, object]:
    # Writes 'debug' — not declared in the contract below.
    return {"documents": [], "debug": {"query": "unexpected"}}


wrapped_fn = contract_node(
    sneaky_retriever,
    NodeContract(
        name="sneaky_retriever",
        reads=("request.text",),
        writes=("documents",),  # 'debug' is NOT declared
        span_kind="RETRIEVER",
    ),
)

try:
    wrapped_fn(INITIAL_STATE)
except StateContractError as exc:
    print("Caught StateContractError:")
    print(f"  contract         : {exc.contract_name}")
    print(f"  undeclared paths : {exc.undeclared_paths}")
    print(f"  message          : {exc}")

---

## Act 2 — send spans to your observability platform

The graph code does not change at all. Only the exporter changes.

Create a `.env` file in this directory with the variables for your platform
(see the recipes below), then run the next two cells.
If `OTEL_EXPORTER_OTLP_ENDPOINT` is unset the cell skips cleanly.

> **Note:** OpenTelemetry's `set_tracer_provider()` is set-once per process.
> If you already ran Act 1 in this kernel, the in-memory provider is installed.
> Restart the kernel, skip Act 1's `configure_local_tracing()` call, and
> run `configure_otlp_tracing()` first instead.

In [ ]:
from dotenv import load_dotenv

from graphobs.demo import configure_otlp_tracing

load_dotenv()  # reads .env from the current directory

# Returns False (with a WARN log) when OTEL_EXPORTER_OTLP_ENDPOINT is unset.
ready = configure_otlp_tracing()
if not ready:
    print("OTLP endpoint not configured — skipping Act 2.")
    print("Add a .env file and restart the kernel to send spans to your platform.")

In [ ]:
if ready:
    # Same graph, same node functions — only the exporter changed.
    act2_result = contract_graph.compile().invoke(dict(INITIAL_STATE))
    print("Spans sent. Open your platform UI to view them.")
    print("Answer:", act2_result.get("answer", {}).get("text"))

---

## Per-platform `.env` recipes

The code above is identical for every platform. Only the `.env` changes.

### Arize Phoenix (cloud)

Full OpenInference support — spans render exactly as designed.

```dotenv
OTEL_EXPORTER_OTLP_ENDPOINT=https://app.phoenix.arize.com/v1/traces
OTEL_EXPORTER_OTLP_HEADERS=api_key=<your-phoenix-api-key>
```

### LangSmith

Ingests OTLP via its own endpoint. OpenInference `input.value`/`output.value`
attributes appear in the generic attributes panel rather than the primary I/O
view — the data is present, just not highlighted.

```dotenv
OTEL_EXPORTER_OTLP_ENDPOINT=https://api.smith.langchain.com/otel/v1/traces
OTEL_EXPORTER_OTLP_HEADERS=x-api-key=<your-langsmith-api-key>
LANGSMITH_PROJECT=<your-project-name>
```

### MLflow (self-hosted)

Start a local MLflow server first: `mlflow server --port 5000`.
OpenInference attributes appear in the span attribute list; the I/O panel
uses MLflow's own convention and will be empty.

```dotenv
OTEL_EXPORTER_OTLP_ENDPOINT=http://localhost:5000/api/2.0/mlflow/otlp/v1/traces
```

### Langfuse

Ingests OTLP. OpenInference attributes are visible in the span detail view.
The `input`/`output` rendering depends on your Langfuse version.

```dotenv
OTEL_EXPORTER_OTLP_ENDPOINT=https://cloud.langfuse.com/api/public/otel/v1/traces
OTEL_EXPORTER_OTLP_HEADERS=Authorization=Basic <base64(public_key:secret_key)>
```